# تدريب SARI

يشغّل `train.py` كما هو. بيانات UCI تُنزَّل هنا. `train_.csv` يُرفع من جهازك في الخلية التالية لأنه مقياس آخر فقط وليس بيانات التدريب.

In [ ]:
!pip -q install torch numpy pandas


In [ ]:
import base64
import os
import urllib.request
import zipfile
from pathlib import Path

ROOT = Path("/content/live_model")
UCI_DIR = Path("/content/sari-client-data/uci")
ROOT.mkdir(parents=True, exist_ok=True)
UCI_DIR.mkdir(parents=True, exist_ok=True)
BLOB = {"train.py": "IyB0b3JjaCBtdXN0IGxvYWQgYmVmb3JlIHBhbmRhcywgb3RoZXJ3aXNlIGMxMC5kbGwgZmFpbHMgdG8gaW5pdGlhbGlzZSBvbiBXaW5kb3dzCmltcG9ydCB0b3JjaAppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gbnVtcHkubGliLnN0cmlkZV90cmlja3MgaW1wb3J0IHNsaWRpbmdfd2luZG93X3ZpZXcKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCmltcG9ydCBzaW0KZnJvbSBtb2RlbCBpbXBvcnQgQ09ORklSTSwgV0FSTl9DT05GSVJNLCBTYXJpTFNUTSwgb3Zlcl9saW1pdCwgcHJlZGljdApmcm9tIHNpbSBpbXBvcnQgRkFVTFRTLCBTSE9SVF9QVSwgV0lORE9XCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApDTElFTlRfVFJBSU4gPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJTQVJJX0NMSUVOVF9UUkFJTiIsIHIiZDpcVlNDb2RlXFByb2plY3RzXHNhcmktY2xpZW50LWRhdGFcdHJhaW5fLmNzdiIpKQpUT0wgPSAzCkdSSUQgPSBucC5yb3VuZChucC5hcmFuZ2UoMC4zMCwgMC45OTEsIDAuMDUpLCAyKQoKCmRlZiBmaXJzdF9hbGFybShwcm9icywgcmF3LCB0YXUpOgogICAgaGl0cyA9IHByb2JzWy4uLiwgOmxlbihGQVVMVFMpXSA+PSBucC5hc2FycmF5KHRhdSkKICAgIGNvbmZpcm0gPSBucC56ZXJvc19saWtlKGhpdHMpCiAgICBjb25maXJtWzosIENPTkZJUk0gLSAxOl0gPSBucC5hbGwobnAuc3RhY2soW2hpdHNbOiwgazpoaXRzLnNoYXBlWzFdIC0gQ09ORklSTSArIDEgKyBrXSBmb3IgayBpbiByYW5nZShDT05GSVJNKV0pLCBheGlzPTApCiAgICBjb25maXJtICY9IG92ZXJfbGltaXQocmF3KQogICAgY29uZmlybVsuLi4sIDBdIHw9IGhpdHNbLi4uLCAwXSAmIChyYXdbLi4uLCAwXSA+PSBTSE9SVF9QVSkKICAgIGZpcmVkID0gY29uZmlybS5hbnkoYXhpcz0xKQogICAgcmV0dXJuIG5wLndoZXJlKGZpcmVkLCBjb25maXJtLmFyZ21heChheGlzPTEpLCAtMSkKCgpkZWYgZmlyc3Rfd2FybmluZyhwcm9icywgdGF1X3ByZSwgbGF0Y2hlZF9hdCk6CiAgICBoaXRzID0gcHJvYnNbLi4uLCAtMV0gPj0gdGF1X3ByZQogICAgb3V0ID0gbnAuZnVsbChsZW4oaGl0cyksIC0xKQogICAgZm9yIG4gaW4gcmFuZ2UobGVuKGhpdHMpKToKICAgICAgICBzdG9wID0gaGl0cy5zaGFwZVsxXSBpZiBsYXRjaGVkX2F0W25dIDwgMCBlbHNlIGxhdGNoZWRfYXRbbl0KICAgICAgICBydW4gPSAwCiAgICAgICAgZm9yIHQgaW4gcmFuZ2Uoc3RvcCk6CiAgICAgICAgICAgIHJ1biA9IHJ1biArIDEgaWYgaGl0c1tuLCB0XSBlbHNlIDAKICAgICAgICAgICAgaWYgcnVuID49IFdBUk5fQ09ORklSTToKICAgICAgICAgICAgICAgIG91dFtuXSA9IHQKICAgICAgICAgICAgICAgIGJyZWFrCiAgICByZXR1cm4gb3V0CgoKZGVmIG5lYXIobGFiZWwsIHQpOgogICAgcmV0dXJuIGxhYmVsW21heCgwLCB0IC0gVE9MKTp0ICsgVE9MICsgMV0uYW55KCkKCgpkZWYgc2NvcmVfc2VxdWVuY2VzKHByb2JzLCB5LCByYXcsIHRhdSk6CiAgICBmaXJzdCA9IGZpcnN0X2FsYXJtKHByb2JzLCByYXcsIHRhdSkKICAgIGZhdWx0ID0geVsuLi4sIDpsZW4oRkFVTFRTKV0gPiAwLjUKICAgIGFueV9mYXVsdCA9IGZhdWx0LmFueShheGlzPTIpCiAgICByZXBvcnQgPSB7fQogICAgZmFsc2VfY3V0ID0gMAogICAgZm9yIG4gaW4gcmFuZ2UobGVuKHkpKToKICAgICAgICBmaXJlZCA9IGZpcnN0W25dW2ZpcnN0W25dID49IDBdCiAgICAgICAgaWYgbGVuKGZpcmVkKSBhbmQgbm90IG5lYXIoYW55X2ZhdWx0W25dLCBpbnQoZmlyZWQubWluKCkpKToKICAgICAgICAgICAgZmFsc2VfY3V0ICs9IDEKICAgIGZvciBrLCBuYW1lIGluIGVudW1lcmF0ZShGQVVMVFMpOgogICAgICAgIGhhcyA9IGZhdWx0WzosIDosIGtdLmFueShheGlzPTEpCiAgICAgICAgb25zZXQgPSBucC53aGVyZShoYXMsIGZhdWx0WzosIDosIGtdLmFyZ21heChheGlzPTEpLCAtMSkKICAgICAgICBoaXQgPSBsYXRlbmN5ID0gd3JvbmcgPSAwCiAgICAgICAgbGF0ID0gW10KICAgICAgICBmb3IgbiBpbiByYW5nZShsZW4oeSkpOgogICAgICAgICAgICB0ID0gZmlyc3Rbbiwga10KICAgICAgICAgICAgaWYgaGFzW25dIGFuZCB0ID49IDAgYW5kIHQgPj0gb25zZXRbbl0gLSBUT0w6CiAgICAgICAgICAgICAgICBoaXQgKz0gMQogICAgICAgICAgICAgICAgbGF0LmFwcGVuZCh0IC0gb25zZXRbbl0pCiAgICAgICAgICAgIGVsaWYgdCA+PSAwIGFuZCBub3QgbmVhcihmYXVsdFtuLCA6LCBrXSwgdCk6CiAgICAgICAgICAgICAgICB3cm9uZyArPSAxCiAgICAgICAgcmVwb3J0W25hbWVdID0gewogICAgICAgICAgICAiZXBpc29kZXMiOiBpbnQoaGFzLnN1bSgpKSwKICAgICAgICAgICAgImRldGVjdGVkIjogcm91bmQoaGl0IC8gbWF4KGludChoYXMuc3VtKCkpLCAxKSwgNCksCiAgICAgICAgICAgICJtZWRpYW5fbGF0ZW5jeV9zIjogZmxvYXQobnAubWVkaWFuKGxhdCkpIGlmIGxhdCBlbHNlIE5vbmUsCiAgICAgICAgICAgICJtYXhfbGF0ZW5jeV9zIjogaW50KG5wLm1heChsYXQpKSBpZiBsYXQgZWxzZSBOb25lLAogICAgICAgICAgICAid3JvbmdfY2xhc3NfYWxhcm1zIjogaW50KHdyb25nKSwKICAgICAgICB9CiAgICByZXBvcnRbImZhbHNlX2N1dG9mZnMiXSA9IGludChmYWxzZV9jdXQpCiAgICByZXBvcnRbInNlcXVlbmNlcyJdID0gaW50KGxlbih5KSkKICAgIHJlcG9ydFsibm9ybWFsX3NlcXVlbmNlcyJdID0gaW50KCh+YW55X2ZhdWx0LmFueShheGlzPTEpKS5zdW0oKSkKICAgIHJldHVybiByZXBvcnQKCgpkZWYgdHJhaW5fbW9kZWwoeCwgeSwgeHYsIHl2LCBlcG9jaHM9MTIpOgogICAgdG9yY2gubWFudWFsX3NlZWQoMCkKICAgIG1vZGVsID0gU2FyaUxTVE0oKQogICAgcG9zID0geS5yZXNoYXBlKC0xLCB5LnNoYXBlWy0xXSkubWVhbihheGlzPTApCiAgICB3ZWlnaHQgPSB0b3JjaC50ZW5zb3IobnAuY2xpcCgoMSAtIHBvcykgLyBucC5tYXhpbXVtKHBvcywgMWUtNiksIDEsIDI1KSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgIGxvc3NfZm4gPSBubi5CQ0VXaXRoTG9naXRzTG9zcyhwb3Nfd2VpZ2h0PXdlaWdodCkKICAgIG9wdCA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj0yZS0zKQogICAgc2NoZWQgPSB0b3JjaC5vcHRpbS5scl9zY2hlZHVsZXIuU3RlcExSKG9wdCwgc3RlcF9zaXplPTUsIGdhbW1hPTAuNCkKICAgIHh0LCB5dCA9IHRvcmNoLmZyb21fbnVtcHkoeCksIHRvcmNoLmZyb21fbnVtcHkoeSkKICAgIHh2dCwgeXZ0ID0gdG9yY2guZnJvbV9udW1weSh4diksIHRvcmNoLmZyb21fbnVtcHkoeXYpCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoZXBvY2hzKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgb3JkZXIgPSB0b3JjaC5yYW5kcGVybShsZW4oeHQpKQogICAgICAgIHRvdGFsID0gMC4wCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbih4dCksIDI1Nik6CiAgICAgICAgICAgIGlkeCA9IG9yZGVyW3N0YXJ0OnN0YXJ0ICsgMjU2XQogICAgICAgICAgICBvcHQuemVyb19ncmFkKCkKICAgICAgICAgICAgbG9zcyA9IGxvc3NfZm4obW9kZWwoeHRbaWR4XSksIHl0W2lkeF0pCiAgICAgICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgICAgICBubi51dGlscy5jbGlwX2dyYWRfbm9ybV8obW9kZWwucGFyYW1ldGVycygpLCAxLjApCiAgICAgICAgICAgIG9wdC5zdGVwKCkKICAgICAgICAgICAgdG90YWwgKz0gbG9zcy5pdGVtKCkgKiBsZW4oaWR4KQogICAgICAgIHNjaGVkLnN0ZXAoKQogICAgICAgIG1vZGVsLmV2YWwoKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICB2YWwgPSBsb3NzX2ZuKG1vZGVsKHh2dFs6NDAwMF0pLCB5dnRbOjQwMDBdKS5pdGVtKCkKICAgICAgICBwcmludChmImVwb2NoIHtlcG9jaCArIDF9IHRyYWluIHt0b3RhbCAvIGxlbih4dCk6LjRmfSB2YWwge3ZhbDouNGZ9IiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBtb2RlbAoKCmRlZiBzdHJlYW1fYXJyYXlzKGZyYW1lLCBybmcsIGRheXMpOgogICAgZnJhbWUgPSBmcmFtZS5kcm9wbmEoKQogICAgZ2FwcyA9IGZyYW1lLmluZGV4LnRvX3NlcmllcygpLmRpZmYoKSAhPSBwZC5UaW1lZGVsdGEobWludXRlcz0xKQogICAgcnVucyA9IGdhcHMuY3Vtc3VtKCkKICAgIHN0cmVhbXMgPSBbXQogICAgdG90YWwgPSAwCiAgICBmb3IgXywgcGFydCBpbiBmcmFtZS5ncm91cGJ5KHJ1bnMpOgogICAgICAgIGlmIGxlbihwYXJ0KSA8IDYwICogNjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBtaW51dGVzID0gbGVuKHBhcnQpCiAgICAgICAgdCA9IG5wLmFyYW5nZShtaW51dGVzICogNjApCiAgICAgICAgaSA9IG5wLmludGVycCh0LCBucC5hcmFuZ2UobWludXRlcykgKiA2MCArIDMwLCBwYXJ0LmkudG9fbnVtcHkoKSkgLyBzaW0uVWNpU291cmNlLlJBVEVECiAgICAgICAgdiA9IG5wLmludGVycCh0LCBucC5hcmFuZ2UobWludXRlcykgKiA2MCArIDMwLCBwYXJ0LnYudG9fbnVtcHkoKSkgLyBmbG9hdChucC5tZWRpYW4oZnJhbWUudikpCiAgICAgICAgaSA9IG5wLmNsaXAoaSAqICgxICsgcm5nLm5vcm1hbCgwLCAwLjAyLCBsZW4odCkpKSwgMCwgTm9uZSkKICAgICAgICB2ID0gdiAqICgxICsgcm5nLm5vcm1hbCgwLCAwLjAwMywgbGVuKHQpKSkKICAgICAgICBsZWFrID0gbnAuY2xpcCgxNSArIDMgKiBpICsgcm5nLnVuaWZvcm0oLTIsIDIsIGxlbih0KSksIDAsIE5vbmUpCiAgICAgICAgdGVtcCA9IHNpbS50aGVybWFsKGksIG5wLnJhbmRvbS5kZWZhdWx0X3JuZygxKSwgYW1iaWVudD0zMC4wKSArIHJuZy51bmlmb3JtKC0wLjMsIDAuMywgbGVuKHQpKQogICAgICAgIHN0cmVhbXMuYXBwZW5kKG5wLnN0YWNrKFtpLCB2LCBsZWFrLCB0ZW1wXSwgYXhpcz0xKS5hc3R5cGUobnAuZmxvYXQzMikpCiAgICAgICAgdG90YWwgKz0gbGVuKHQpCiAgICAgICAgaWYgdG90YWwgPj0gZGF5cyAqIDg2NDAwOgogICAgICAgICAgICBicmVhawogICAgcmV0dXJuIHN0cmVhbXMKCgpkZWYgc3RyZWFtX3Byb2JzKG1vZGVsLCByYXcpOgogICAgZmVhdHMgPSBzaW0uZmVhdHVyZXMocmF3WzosIDBdLCByYXdbOiwgMV0sIHJhd1s6LCAyXSwgcmF3WzosIDNdKQogICAgd2luZG93cyA9IHNsaWRpbmdfd2luZG93X3ZpZXcoZmVhdHMsIChXSU5ET1csIDQpKVs6LCAwXQogICAgbW9kZWwuZXZhbCgpCiAgICBvdXRzID0gW10KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBzdGFydCBpbiByYW5nZSgwLCBsZW4od2luZG93cyksIDQwOTYpOgogICAgICAgICAgICBjaHVuayA9IHRvcmNoLmZyb21fbnVtcHkobnAuYXNjb250aWd1b3VzYXJyYXkod2luZG93c1tzdGFydDpzdGFydCArIDQwOTZdKSkKICAgICAgICAgICAgb3V0cy5hcHBlbmQodG9yY2guc2lnbW9pZChtb2RlbChjaHVuaylbOiwgLTFdKS5udW1weSgpKQogICAgcHJvYnMgPSBucC5jb25jYXRlbmF0ZShvdXRzKQogICAgcmV0dXJuIHByb2JzLCByYXdbV0lORE9XIC0gMTpdCgoKZGVmIHN0cmVhbV9hbGFybXMocHJvYnMsIHJhdywgdGF1KToKICAgIGhpdHMgPSBwcm9ic1s6LCA6bGVuKEZBVUxUUyldID49IG5wLmFzYXJyYXkodGF1KQogICAgY29uZmlybSA9IG5wLnplcm9zX2xpa2UoaGl0cykKICAgIGNvbmZpcm1bQ09ORklSTSAtIDE6XSA9IG5wLmFsbChucC5zdGFjayhbaGl0c1trOmxlbihoaXRzKSAtIENPTkZJUk0gKyAxICsga10gZm9yIGsgaW4gcmFuZ2UoQ09ORklSTSldKSwgYXhpcz0wKQogICAgY29uZmlybSAmPSBvdmVyX2xpbWl0KHJhdykKICAgIGNvbmZpcm1bOiwgMF0gfD0gaGl0c1s6LCAwXSAmIChyYXdbOiwgMF0gPj0gU0hPUlRfUFUpCiAgICByZXR1cm4gY29uZmlybQoKCmRlZiBhcHBfc3RyZWFtKHJuZywgc2Vjb25kcywgbW9kZT0oKSk6CiAgICBpID0gMTQuNSArIHJuZy51bmlmb3JtKC0wLjYsIDAuNiwgc2Vjb25kcykKICAgIHYgPSAyMjAuMCArIHJuZy51bmlmb3JtKC0yLCAyLCBzZWNvbmRzKQogICAgbGVhayA9IDAuMDIgKyBybmcudW5pZm9ybSgtMC4wMDIsIDAuMDAyLCBzZWNvbmRzKQogICAgdGVtcCA9IDMyLjAgKyBybmcudW5pZm9ybSgtMC4zLCAwLjMsIHNlY29uZHMpCiAgICBpZiAic3VyZ2UiIGluIG1vZGU6CiAgICAgICAgaSA9IDQ4LjUgKyBybmcudW5pZm9ybSgwLCA0LCBzZWNvbmRzKQogICAgICAgIHRlbXAgPSB0ZW1wICsgOC4wCiAgICBpZiAiZGlwIiBpbiBtb2RlOgogICAgICAgIHYgPSAxNzIuMCArIHJuZy51bmlmb3JtKDAsIDUsIHNlY29uZHMpCiAgICBpZiAibGVhayIgaW4gbW9kZToKICAgICAgICBsZWFrID0gMC40MiArIHJuZy51bmlmb3JtKDAsIDAuMSwgc2Vjb25kcykKICAgIHJldHVybiBucC5zdGFjayhbaSAvIDQwLjAsIHYgLyAyMjAuMCwgbGVhayAqIDEwMDAuMCwgdGVtcF0sIGF4aXM9MSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIGFwcF9zY2VuYXJpb3MobW9kZWwsIHRhdSk6CiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoNykKICAgIG5vcm1hbCA9IGFwcF9zdHJlYW0ocm5nLCA4NjQwMCkKICAgIHByb2JzLCByYXcgPSBzdHJlYW1fcHJvYnMobW9kZWwsIG5vcm1hbCkKICAgIG91dCA9IHsibm9ybWFsXzI0aF9mYWxzZV9hbGFybXMiOiBpbnQoc3RyZWFtX2FsYXJtcyhwcm9icywgcmF3LCB0YXUpLmFueShheGlzPTEpLnN1bSgpKX0KICAgIGZvciBuYW1lLCBtb2RlIGluICgoInN1cmdlIiwgKCJzdXJnZSIsKSksICgiZGlwIiwgKCJkaXAiLCkpLCAoImxlYWsiLCAoImxlYWsiLCkpLCAoInN1cmdlK2xlYWsiLCAoInN1cmdlIiwgImxlYWsiKSkpOgogICAgICAgIHN0cmVhbSA9IG5wLmNvbmNhdGVuYXRlKFthcHBfc3RyZWFtKHJuZywgMzAwKSwgYXBwX3N0cmVhbShybmcsIDYwLCBtb2RlKV0pCiAgICAgICAgcHJvYnMsIHJhdyA9IHN0cmVhbV9wcm9icyhtb2RlbCwgc3RyZWFtKQogICAgICAgIGNvbmZpcm0gPSBzdHJlYW1fYWxhcm1zKHByb2JzLCByYXcsIHRhdSkKICAgICAgICBvbnNldCA9IDMwMCAtIChXSU5ET1cgLSAxKQogICAgICAgIGZpcmVkID0ge0ZBVUxUU1trXTogKGludChucC5hcmdtYXgoY29uZmlybVs6LCBrXSkpIC0gb25zZXQgaWYgY29uZmlybVs6LCBrXS5hbnkoKSBlbHNlIE5vbmUpIGZvciBrIGluIHJhbmdlKGxlbihGQVVMVFMpKX0KICAgICAgICBvdXRbbmFtZV0gPSB7CiAgICAgICAgICAgICJiZWZvcmVfZmF1bHRfYWxhcm1zIjogaW50KGNvbmZpcm1bOm9uc2V0XS5hbnkoYXhpcz0xKS5zdW0oKSksCiAgICAgICAgICAgICJsYXRlbmN5X3MiOiB7a2V5OiB2YWwgZm9yIGtleSwgdmFsIGluIGZpcmVkLml0ZW1zKCkgaWYgdmFsIGlzIG5vdCBOb25lfSwKICAgICAgICB9CiAgICByZXR1cm4gb3V0CgoKZGVmIHR1bmUocHJvYnMsIHksIHJhdywgdmFsX3N0cmVhbSk6CiAgICB0YXUgPSBbXQogICAgZm9yIGssIG5hbWUgaW4gZW51bWVyYXRlKEZBVUxUUyk6CiAgICAgICAgYmVzdCA9IE5vbmUKICAgICAgICBmb3IgdCBpbiBHUklEOgogICAgICAgICAgICB0cmlhbCA9IFsxLjAxXSAqIGxlbihGQVVMVFMpCiAgICAgICAgICAgIHRyaWFsW2tdID0gdAogICAgICAgICAgICByZXAgPSBzY29yZV9zZXF1ZW5jZXMocHJvYnMsIHksIHJhdywgdHJpYWwpCiAgICAgICAgICAgIHN0cmVhbV9mcCA9IHN1bShpbnQoc3RyZWFtX2FsYXJtcyhwLCByLCB0cmlhbClbOiwga10uc3VtKCkpIGZvciBwLCByIGluIHZhbF9zdHJlYW0pCiAgICAgICAgICAgIGZwID0gcmVwWyJmYWxzZV9jdXRvZmZzIl0gKyByZXBbbmFtZV1bIndyb25nX2NsYXNzX2FsYXJtcyJdICsgc3RyZWFtX2ZwCiAgICAgICAgICAgIGtleSA9IChmcCwgLXJlcFtuYW1lXVsiZGV0ZWN0ZWQiXSwgdCkKICAgICAgICAgICAgaWYgYmVzdCBpcyBOb25lIG9yIGtleSA8IGJlc3RbMF06CiAgICAgICAgICAgICAgICBiZXN0ID0gKGtleSwgZmxvYXQodCkpCiAgICAgICAgdGF1LmFwcGVuZChiZXN0WzFdKQogICAgICAgIHByaW50KGYidGF1IHtuYW1lfSA9IHtiZXN0WzFdfSAoZmFsc2Uge2Jlc3RbMF1bMF19LCBkZXRlY3RlZCB7LWJlc3RbMF1bMV19KSIsIGZsdXNoPVRydWUpCiAgICBsYXRjaGVkID0gZmlyc3RfYWxhcm0ocHJvYnMsIHJhdywgdGF1KQogICAgbGF0Y2hlZCA9IG5wLndoZXJlKChsYXRjaGVkID49IDApLmFueShheGlzPTEpLCBucC53aGVyZShsYXRjaGVkID49IDAsIGxhdGNoZWQsIDEwICoqIDYpLm1pbihheGlzPTEpLCAtMSkKICAgIGFueV9mYXVsdCA9ICh5Wy4uLiwgOmxlbihGQVVMVFMpXSA+IDAuNSkuYW55KGF4aXM9MikKICAgIHRhdV9wcmUgPSAxLjAxCiAgICBmb3IgdCBpbiBHUklEOgogICAgICAgIHdhcm4gPSBmaXJzdF93YXJuaW5nKHByb2JzLCB0LCBsYXRjaGVkKQogICAgICAgIGZhbHNlID0gc3VtKDEgZm9yIG4gaW4gcmFuZ2UobGVuKHkpKSBpZiB3YXJuW25dID49IDAgYW5kIG5vdCBhbnlfZmF1bHRbbiwgd2FybltuXTp3YXJuW25dICsgc2ltLkhPUklaT04gKyBUT0xdLmFueSgpKQogICAgICAgIGlmIGZhbHNlID09IDA6CiAgICAgICAgICAgIHRhdV9wcmUgPSBmbG9hdCh0KQogICAgICAgICAgICBicmVhawogICAgcmV0dXJuIHRhdSwgdGF1X3ByZQoKCmRlZiBtYWluKCk6CiAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMob3MuY3B1X2NvdW50KCkgb3IgNCkKICAgIHN0YXJ0ZWQgPSB0aW1lLnRpbWUoKQogICAgcGFydHMgPSBzaW0uc3BsaXRfdWNpKHNpbS5sb2FkX3VjaSgpKQogICAgc291cmNlcyA9IHtuYW1lOiBzaW0uVWNpU291cmNlKGZyYW1lKSBmb3IgbmFtZSwgZnJhbWUgaW4gcGFydHMuaXRlbXMoKX0KICAgIHgsIHksIF8gPSBzaW0uYmF0Y2goODAwMDAgaWYgIi0tZXZhbCIgbm90IGluIHN5cy5hcmd2IGVsc2UgMTAsIDEsIHNvdXJjZXNbInRyYWluIl0pCiAgICB4diwgeXYsIHJ2ID0gc2ltLmJhdGNoKDgwMDAsIDIsIHNvdXJjZXNbInZhbCJdKQogICAgeHMsIHlzLCBycyA9IHNpbS5iYXRjaCgyMDAwMCwgMywgc291cmNlc1sidGVzdCJdKQogICAgcHJpbnQoZiJkYXRhIHJlYWR5IGluIHt0aW1lLnRpbWUoKSAtIHN0YXJ0ZWQ6LjBmfXMiLCBmbHVzaD1UcnVlKQogICAgaWYgIi0tZXZhbCIgaW4gc3lzLmFyZ3Y6CiAgICAgICAgbW9kZWwgPSBTYXJpTFNUTSgpCiAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQoUk9PVCAvICJzYXJpX2xzdG0ucHQiKSkKICAgIGVsc2U6CiAgICAgICAgbW9kZWwgPSB0cmFpbl9tb2RlbCh4LCB5LCB4diwgeXYpCiAgICAgICAgdG9yY2guc2F2ZShtb2RlbC5zdGF0ZV9kaWN0KCksIFJPT1QgLyAic2FyaV9sc3RtLnB0IikKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygxMSkKICAgIHZhbF9zdHJlYW0gPSBbc3RyZWFtX3Byb2JzKG1vZGVsLCBzKSBmb3IgcyBpbiBzdHJlYW1fYXJyYXlzKHBhcnRzWyJ2YWwiXSwgcm5nLCAzKV0KICAgIHRhdSwgdGF1X3ByZSA9IHR1bmUocHJlZGljdChtb2RlbCwgeHYpLCB5diwgcnYsIHZhbF9zdHJlYW0pCiAgICBjb25maWcgPSB7CiAgICAgICAgIndpbmRvdyI6IFdJTkRPVywKICAgICAgICAidGF1IjogdGF1LAogICAgICAgICJ0YXVfcHJlIjogdGF1X3ByZSwKICAgICAgICAiY29uZmlybSI6IENPTkZJUk0sCiAgICAgICAgImxpbWl0cyI6IHsKICAgICAgICAgICAgIm92ZXJsb2FkX3B1Ijogc2ltLk9WRVJMT0FEX1BVLAogICAgICAgICAgICAic2hvcnRfcHUiOiBTSE9SVF9QVSwKICAgICAgICAgICAgInZfbG93X3B1Ijogc2ltLlZfTE9XLAogICAgICAgICAgICAidl9oaWdoX3B1Ijogc2ltLlZfSElHSCwKICAgICAgICAgICAgImxlYWtfbWEiOiBzaW0uTEVBS19NQSwKICAgICAgICAgICAgInRlbXBfYyI6IHNpbS5URU1QX0MsCiAgICAgICAgICAgICJwZXJzaXN0X3MiOiBzaW0uUEVSU0lTVCwKICAgICAgICB9LAogICAgfQogICAgKFJPT1QgLyAic2FyaV9jb25maWcuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhjb25maWcsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHRlc3RfcHJvYnMgPSBwcmVkaWN0KG1vZGVsLCB4cykKICAgIHJlYWxfdGVzdCA9IFtzdHJlYW1fcHJvYnMobW9kZWwsIHMpIGZvciBzIGluIHN0cmVhbV9hcnJheXMocGFydHNbInRlc3QiXSwgbnAucmFuZG9tLmRlZmF1bHRfcm5nKDEyKSwgMjApXQogICAgcmVhbF9zZWNvbmRzID0gc3VtKGxlbihwKSBmb3IgcCwgXyBpbiByZWFsX3Rlc3QpCiAgICByZWFsX2ZhbHNlID0gc3VtKGludChzdHJlYW1fYWxhcm1zKHAsIHIsIHRhdSkuYW55KGF4aXM9MSkuc3VtKCkpIGZvciBwLCByIGluIHJlYWxfdGVzdCkKICAgIGNsaWVudCA9IHBkLnJlYWRfY3N2KENMSUVOVF9UUkFJTiwgdXNlY29scz1bImFub21hbHkiXSkuYW5vbWFseQogICAgbWV0cmljcyA9IHsKICAgICAgICAidGVzdF9zZXF1ZW5jZXMiOiBzY29yZV9zZXF1ZW5jZXModGVzdF9wcm9icywgeXMsIHJzLCB0YXUpLAogICAgICAgICJyZWFsX3VjaV9oZWxkX291dCI6IHsiZGF5cyI6IHJvdW5kKHJlYWxfc2Vjb25kcyAvIDg2NDAwLCAxKSwgImZhbHNlX2FsYXJtX3NlY29uZHMiOiByZWFsX2ZhbHNlfSwKICAgICAgICAiYXBwX3NpbXVsYXRpb24iOiBhcHBfc2NlbmFyaW9zKG1vZGVsLCB0YXUpLAogICAgICAgICJjbGllbnRfZGF0YXNldF9hbGxfbm9ybWFsX2FjY3VyYWN5Ijogcm91bmQoZmxvYXQoMSAtIGNsaWVudC5tZWFuKCkpLCA0KSwKICAgICAgICAidGF1IjogdGF1LAogICAgICAgICJ0YXVfcHJlIjogdGF1X3ByZSwKICAgICAgICAibWludXRlcyI6IHJvdW5kKCh0aW1lLnRpbWUoKSAtIHN0YXJ0ZWQpIC8gNjAsIDEpLAogICAgfQogICAgKFJPT1QgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KGpzb24uZHVtcHMobWV0cmljcywgaW5kZW50PTIpLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "model.py": "aW1wb3J0IHRvcmNoCmltcG9ydCBudW1weSBhcyBucApmcm9tIHRvcmNoIGltcG9ydCBubgoKZnJvbSBzaW0gaW1wb3J0IEZBVUxUUywgTEVBS19NQSwgT1ZFUkxPQURfUFUsIFNIT1JUX1BVLCBURU1QX0MsIFZfSElHSCwgVl9MT1cKCkNPTkZJUk0gPSAzCldBUk5fQ09ORklSTSA9IDUKCgpkZWYgb3Zlcl9saW1pdChyYXcpOgogICAgcmV0dXJuIG5wLnN0YWNrKFsKICAgICAgICByYXdbLi4uLCAwXSA+PSBPVkVSTE9BRF9QVSwKICAgICAgICAocmF3Wy4uLiwgMV0gPCBWX0xPVykgfCAocmF3Wy4uLiwgMV0gPiBWX0hJR0gpLAogICAgICAgIHJhd1suLi4sIDJdID49IExFQUtfTUEsCiAgICAgICAgcmF3Wy4uLiwgM10gPj0gVEVNUF9DLAogICAgXSwgYXhpcz0tMSkKCgpjbGFzcyBTYXJpTFNUTShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGhpZGRlbj02NCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5sc3RtID0gbm4uTFNUTSg0LCBoaWRkZW4sIG51bV9sYXllcnM9MiwgYmF0Y2hfZmlyc3Q9VHJ1ZSwgZHJvcG91dD0wLjEpCiAgICAgICAgc2VsZi5oZWFkID0gbm4uTGluZWFyKGhpZGRlbiwgbGVuKEZBVUxUUykgKyAxKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIG91dCwgXyA9IHNlbGYubHN0bSh4KQogICAgICAgIHJldHVybiBzZWxmLmhlYWQob3V0KQoKCmRlZiBwcmVkaWN0KG1vZGVsLCB4LCBiYXRjaF9zaXplPTQwOTYpOgogICAgbW9kZWwuZXZhbCgpCiAgICBvdXRzID0gW10KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBzdGFydCBpbiByYW5nZSgwLCBsZW4oeCksIGJhdGNoX3NpemUpOgogICAgICAgICAgICBjaHVuayA9IHRvcmNoLmZyb21fbnVtcHkoeFtzdGFydDpzdGFydCArIGJhdGNoX3NpemVdKQogICAgICAgICAgICBvdXRzLmFwcGVuZCh0b3JjaC5zaWdtb2lkKG1vZGVsKGNodW5rKSkubnVtcHkoKSkKICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShvdXRzKQoKCmNsYXNzIEFsYXJtUG9saWN5OgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRhdSwgdGF1X3ByZSk6CiAgICAgICAgc2VsZi50YXUgPSBucC5hc2FycmF5KHRhdSwgZHR5cGU9ZmxvYXQpCiAgICAgICAgc2VsZi50YXVfcHJlID0gZmxvYXQodGF1X3ByZSkKICAgICAgICBzZWxmLnJlc2V0KCkKCiAgICBkZWYgcmVzZXQoc2VsZik6CiAgICAgICAgc2VsZi5jb3VudCA9IG5wLnplcm9zKGxlbihGQVVMVFMpLCBkdHlwZT1pbnQpCiAgICAgICAgc2VsZi53YXJuID0gMAogICAgICAgIHNlbGYubGF0Y2hlZCA9IG5wLnplcm9zKGxlbihGQVVMVFMpLCBkdHlwZT1ib29sKQoKICAgIGRlZiBzdGVwKHNlbGYsIHByb2JzLCBwcmUsIHJhd19yb3cpOgogICAgICAgIGhpdHMgPSBwcm9icyA+PSBzZWxmLnRhdQogICAgICAgIHNlbGYuY291bnQgPSBucC53aGVyZShoaXRzLCBzZWxmLmNvdW50ICsgMSwgMCkKICAgICAgICBjb25maXJtZWQgPSAoc2VsZi5jb3VudCA+PSBDT05GSVJNKSAmIG92ZXJfbGltaXQobnAuYXNhcnJheShyYXdfcm93KSkKICAgICAgICBjb25maXJtZWRbMF0gfD0gaGl0c1swXSBhbmQgcmF3X3Jvd1swXSA+PSBTSE9SVF9QVQogICAgICAgIHJpc2luZyA9IGJvb2woY29uZmlybWVkLmFueSgpIGFuZCBub3Qgc2VsZi5sYXRjaGVkLmFueSgpKQogICAgICAgIHNlbGYubGF0Y2hlZCB8PSBjb25maXJtZWQKICAgICAgICBzZWxmLndhcm4gPSBzZWxmLndhcm4gKyAxIGlmIHByZSA+PSBzZWxmLnRhdV9wcmUgYW5kIG5vdCBzZWxmLmxhdGNoZWQuYW55KCkgZWxzZSAwCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImFsYXJtIjogYm9vbChzZWxmLmxhdGNoZWQuYW55KCkpLAogICAgICAgICAgICAicmlzaW5nIjogcmlzaW5nLAogICAgICAgICAgICAiZmF1bHRzIjogW0ZBVUxUU1trXSBmb3IgayBpbiBucC5ub256ZXJvKHNlbGYubGF0Y2hlZClbMF1dLAogICAgICAgICAgICAid2FybmluZyI6IHNlbGYud2FybiA+PSBXQVJOX0NPTkZJUk0sCiAgICAgICAgfQoKCmRlZiBydW5fcG9saWN5KHByb2JzLCByYXcsIHRhdSwgdGF1X3ByZSk6CiAgICBwb2xpY3kgPSBBbGFybVBvbGljeSh0YXUsIHRhdV9wcmUpCiAgICBmaXJzdCA9IG5wLmZ1bGwobGVuKEZBVUxUUyksIC0xKQogICAgd2Fybl9maXJzdCA9IC0xCiAgICBmb3IgdCBpbiByYW5nZShwcm9icy5zaGFwZVswXSk6CiAgICAgICAgZGVjaXNpb24gPSBwb2xpY3kuc3RlcChwcm9ic1t0LCA6bGVuKEZBVUxUUyldLCBwcm9ic1t0LCAtMV0sIHJhd1t0XSkKICAgICAgICBmb3IgbmFtZSBpbiBkZWNpc2lvblsiZmF1bHRzIl06CiAgICAgICAgICAgIGsgPSBGQVVMVFMuaW5kZXgobmFtZSkKICAgICAgICAgICAgaWYgZmlyc3Rba10gPCAwOgogICAgICAgICAgICAgICAgZmlyc3Rba10gPSB0CiAgICAgICAgaWYgZGVjaXNpb25bIndhcm5pbmciXSBhbmQgd2Fybl9maXJzdCA8IDA6CiAgICAgICAgICAgIHdhcm5fZmlyc3QgPSB0CiAgICByZXR1cm4gZmlyc3QsIHdhcm5fZmlyc3QK", "sim.py": "aW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKClVDSV9QQVRIID0gUGF0aChvcy5lbnZpcm9uLmdldCgiU0FSSV9VQ0lfUEFUSCIsIHIiZDpcVlNDb2RlXFByb2plY3RzXHNhcmktY2xpZW50LWRhdGFcdWNpXGhvdXNlaG9sZF9wb3dlcl9jb25zdW1wdGlvbi50eHQiKSkKCk9WRVJMT0FEX1BVID0gMS4xMwpTSE9SVF9QVSA9IDUuMApWX0xPVyA9IDAuOTAKVl9ISUdIID0gMS4xMApMRUFLX01BID0gMzAuMApURU1QX0MgPSA3MC4wClBFUlNJU1QgPSAzCkhPUklaT04gPSA2MApXSU5ET1cgPSAxMjAKRkFVTFRTID0gKCJvdmVyY3VycmVudCIsICJ2b2x0YWdlIiwgImxlYWthZ2UiLCAib3ZlcmhlYXQiKQoKCmRlZiBsb2FkX3VjaShwYXRoPVVDSV9QQVRIKToKICAgIGRmID0gcGQucmVhZF9jc3YocGF0aCwgc2VwPSI7IiwgbmFfdmFsdWVzPSI/IiwgdXNlY29scz1bIkRhdGUiLCAiVGltZSIsICJWb2x0YWdlIiwgIkdsb2JhbF9pbnRlbnNpdHkiXSwgbG93X21lbW9yeT1GYWxzZSkKICAgIGluZGV4ID0gcGQudG9fZGF0ZXRpbWUoZGYuRGF0ZSArICIgIiArIGRmLlRpbWUsIGZvcm1hdD0iJWQvJW0vJVkgJUg6JU06JVMiKQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZSh7InYiOiBkZi5Wb2x0YWdlLnRvX251bXB5KGZsb2F0KSwgImkiOiBkZi5HbG9iYWxfaW50ZW5zaXR5LnRvX251bXB5KGZsb2F0KX0sIGluZGV4PWluZGV4KQoKCmRlZiBzcGxpdF91Y2koZnJhbWUpOgogICAgcmV0dXJuIHsKICAgICAgICAidHJhaW4iOiBmcmFtZVtmcmFtZS5pbmRleCA8ICIyMDEwLTAxLTAxIl0sCiAgICAgICAgInZhbCI6IGZyYW1lWyhmcmFtZS5pbmRleCA+PSAiMjAxMC0wMS0wMSIpICYgKGZyYW1lLmluZGV4IDwgIjIwMTAtMDYtMDEiKV0sCiAgICAgICAgInRlc3QiOiBmcmFtZVtmcmFtZS5pbmRleCA+PSAiMjAxMC0wNi0wMSJdLAogICAgfQoKCmNsYXNzIFVjaVNvdXJjZToKICAgIFJBVEVEID0gNjAuMAoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBmcmFtZSwgc3Bhbj01KToKICAgICAgICBzZWxmLnYgPSBmcmFtZS52LnRvX251bXB5KCkKICAgICAgICBzZWxmLmkgPSBmcmFtZS5pLnRvX251bXB5KCkKICAgICAgICBzZWxmLnZfcmVmID0gZmxvYXQobnAubmFubWVkaWFuKHNlbGYudikpCiAgICAgICAgYmFkID0gbnAuaXNuYW4oc2VsZi52KSB8IG5wLmlzbmFuKHNlbGYuaSkKICAgICAgICBjb3VudHMgPSBucC5jb25jYXRlbmF0ZShbWzBdLCBucC5jdW1zdW0oYmFkKV0pCiAgICAgICAgc2VsZi5zcGFuID0gc3BhbgogICAgICAgIHNlbGYuc3RhcnRzID0gbnAubm9uemVybyhjb3VudHNbc3BhbjpdIC0gY291bnRzWzotc3Bhbl0gPT0gMClbMF0KCiAgICBkZWYgc2Vjb25kcyhzZWxmLCBybmcsIGxlbmd0aCk6CiAgICAgICAgc3RhcnQgPSBpbnQocm5nLmNob2ljZShzZWxmLnN0YXJ0cykpCiAgICAgICAgbWludXRlcyA9IG5wLmFyYW5nZShzZWxmLnNwYW4pICogNjAuMCArIDMwLjAKICAgICAgICBvZmZzZXQgPSBybmcudW5pZm9ybSgwLCA2MCkKICAgICAgICB0ID0gbnAuYXJhbmdlKGxlbmd0aCkgKyBvZmZzZXQKICAgICAgICBpID0gbnAuaW50ZXJwKHQsIG1pbnV0ZXMsIHNlbGYuaVtzdGFydDpzdGFydCArIHNlbGYuc3Bhbl0pIC8gc2VsZi5SQVRFRAogICAgICAgIHYgPSBucC5pbnRlcnAodCwgbWludXRlcywgc2VsZi52W3N0YXJ0OnN0YXJ0ICsgc2VsZi5zcGFuXSkgLyBzZWxmLnZfcmVmCiAgICAgICAgcmV0dXJuIGksIHYKCgpkZWYgdGhlcm1hbChpX3B1LCBybmcsIGFtYmllbnQ9Tm9uZSk6CiAgICBhbWJpZW50ID0gcm5nLnVuaWZvcm0oMTgsIDQyKSBpZiBhbWJpZW50IGlzIE5vbmUgZWxzZSBhbWJpZW50CiAgICByaXNlID0gcm5nLnVuaWZvcm0oMTAsIDMwKQogICAgdGF1ID0gcm5nLnVuaWZvcm0oMTIwLCA5MDApCiAgICB0ZW1wID0gbnAuZW1wdHlfbGlrZShpX3B1KQogICAgdGVtcFswXSA9IGFtYmllbnQgKyByaXNlICogaV9wdVswXSAqKiAyCiAgICBmb3IgdCBpbiByYW5nZSgxLCBsZW4oaV9wdSkpOgogICAgICAgIHRhcmdldCA9IGFtYmllbnQgKyByaXNlICogaV9wdVt0XSAqKiAyCiAgICAgICAgdGVtcFt0XSA9IHRlbXBbdCAtIDFdICsgKHRhcmdldCAtIHRlbXBbdCAtIDFdKSAvIHRhdQogICAgcmV0dXJuIHRlbXAKCgpkZWYgX3NwYW4ocm5nLCBsZW5ndGgsIHN0YXJ0LCByYW1wKToKICAgIHNoYXBlID0gbnAuemVyb3MobGVuZ3RoKQogICAgaWYgcmFtcDoKICAgICAgICBkdXIgPSBpbnQocm5nLmludGVnZXJzKDIwLCAxNTApKQogICAgICAgIHN0b3AgPSBtaW4obGVuZ3RoLCBzdGFydCArIGR1cikKICAgICAgICBzaGFwZVtzdGFydDpzdG9wXSA9IG5wLmxpbnNwYWNlKDAsIDEsIHN0b3AgLSBzdGFydCwgZW5kcG9pbnQ9RmFsc2UpCiAgICAgICAgc2hhcGVbc3RvcDpdID0gMS4wCiAgICBlbHNlOgogICAgICAgIHNoYXBlW3N0YXJ0Ol0gPSAxLjAKICAgIHJldHVybiBzaGFwZQoKCmRlZiBpbmplY3Qoa2luZCwgcmF3LCBybmcsIHN0YXJ0LCBhcHBfc3R5bGUpOgogICAgaSwgdiwgbGVhaywgdGVtcCA9IHJhdwogICAgbGVuZ3RoID0gbGVuKGkpCiAgICByYW1wID0gKG5vdCBhcHBfc3R5bGUpIGFuZCBybmcucmFuZG9tKCkgPCAwLjQKICAgIHNoYXBlID0gX3NwYW4ocm5nLCBsZW5ndGgsIHN0YXJ0LCByYW1wKQogICAgaWYga2luZCA9PSAib3ZlcmN1cnJlbnQiOgogICAgICAgIGlmIGFwcF9zdHlsZToKICAgICAgICAgICAgbGV2ZWwgPSAoNDguNSArIHJuZy51bmlmb3JtKDAsIDQsIGxlbmd0aCkpIC8gNDAuMAogICAgICAgICAgICB0ZW1wICs9IDguMCAqIHNoYXBlCiAgICAgICAgZWxpZiBybmcucmFuZG9tKCkgPCAwLjE1OgogICAgICAgICAgICBsZXZlbCA9IHJuZy51bmlmb3JtKFNIT1JUX1BVLCAxMi4wKSAqICgxICsgcm5nLm5vcm1hbCgwLCAwLjAyLCBsZW5ndGgpKQogICAgICAgICAgICBzaGFwZSA9IF9zcGFuKHJuZywgbGVuZ3RoLCBzdGFydCwgRmFsc2UpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbGV2ZWwgPSBybmcudW5pZm9ybSgxLjIsIDMuMCkgKiAoMSArIHJuZy5ub3JtYWwoMCwgMC4wMiwgbGVuZ3RoKSkKICAgICAgICBpWzpdID0gaSArIHNoYXBlICogKGxldmVsIC0gaSkKICAgIGVsaWYga2luZCA9PSAidm9sdGFnZSI6CiAgICAgICAgaWYgYXBwX3N0eWxlOgogICAgICAgICAgICBsZXZlbCA9ICgxNzIuMCArIHJuZy51bmlmb3JtKDAsIDUsIGxlbmd0aCkpIC8gMjIwLjAKICAgICAgICBlbGlmIHJuZy5yYW5kb20oKSA8IDAuNToKICAgICAgICAgICAgbGV2ZWwgPSBybmcudW5pZm9ybSgwLjA1LCAwLjg2KSArIHJuZy5ub3JtYWwoMCwgMC4wMDQsIGxlbmd0aCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBsZXZlbCA9IHJuZy51bmlmb3JtKDEuMTQsIDEuMzUpICsgcm5nLm5vcm1hbCgwLCAwLjAwNCwgbGVuZ3RoKQogICAgICAgIHZbOl0gPSB2ICsgc2hhcGUgKiAobGV2ZWwgLSB2KQogICAgZWxpZiBraW5kID09ICJsZWFrYWdlIjoKICAgICAgICBpZiBhcHBfc3R5bGU6CiAgICAgICAgICAgIGxldmVsID0gNDIwLjAgKyBybmcudW5pZm9ybSgwLCAxMDAsIGxlbmd0aCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBsZXZlbCA9IHJuZy51bmlmb3JtKDQwLCA4MDApICsgcm5nLnVuaWZvcm0oLTIsIDIsIGxlbmd0aCkKICAgICAgICBsZWFrWzpdID0gbGVhayArIHNoYXBlICogKGxldmVsIC0gbGVhaykKICAgIGVsc2U6CiAgICAgICAgcGVhayA9IHJuZy51bmlmb3JtKDcyLCAxMTApCiAgICAgICAgc2hhcGUgPSBfc3BhbihybmcsIGxlbmd0aCwgc3RhcnQsIFRydWUpCiAgICAgICAgdGVtcFs6XSA9IHRlbXAgKyBzaGFwZSAqIChwZWFrIC0gdGVtcCkKCgpkZWYgZGlzdHJhY3QocmF3LCBybmcsIGFwcF9zdHlsZSk6CiAgICBpLCB2LCBsZWFrLCB0ZW1wID0gcmF3CiAgICBsZW5ndGggPSBsZW4oaSkKICAgIGZvciBfIGluIHJhbmdlKGludChybmcuaW50ZWdlcnMoMSwgNSkpKToKICAgICAgICBhdCA9IGludChybmcuaW50ZWdlcnMoMCwgbGVuZ3RoIC0gMikpCiAgICAgICAgZHVyID0gaW50KHJuZy5pbnRlZ2VycygxLCBQRVJTSVNUKSkKICAgICAgICBraW5kID0gcm5nLmludGVnZXJzKDAsIDUpCiAgICAgICAgaWYga2luZCA9PSAwOgogICAgICAgICAgICBpW2F0OmF0ICsgZHVyXSA9IG5wLm1pbmltdW0oaVthdDphdCArIGR1cl0gKiBybmcudW5pZm9ybSgxLjUsIDQuMCkgKyBybmcudW5pZm9ybSgwLjMsIDEuMiksIDQuNSkKICAgICAgICBlbGlmIGtpbmQgPT0gMToKICAgICAgICAgICAgdlthdDphdCArIGR1cl0gKj0gcm5nLnVuaWZvcm0oMC43NSwgMC45NSkKICAgICAgICBlbGlmIGtpbmQgPT0gMjoKICAgICAgICAgICAgbGVha1thdDphdCArIGR1cl0gKz0gcm5nLnVuaWZvcm0oMTAsIDYwKQogICAgICAgIGVsaWYga2luZCA9PSAzOgogICAgICAgICAgICB0ZW1wW2F0OmF0ICsgZHVyXSArPSBybmcudW5pZm9ybSg1LCA0MCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBob2xkID0gbWluKGxlbmd0aCwgYXQgKyBpbnQocm5nLmludGVnZXJzKDEsIDYpKSkKICAgICAgICAgICAgZm9yIGFyciBpbiAoaSwgdiwgbGVhaywgdGVtcCk6CiAgICAgICAgICAgICAgICBhcnJbYXQ6aG9sZF0gPSBhcnJbYXQgLSAxXSBpZiBhdCA+IDAgZWxzZSBhcnJbMF0KCgpkZWYgYXBwX2xpa2Uocm5nLCBsZW5ndGgpOgogICAgaSA9IHJuZy51bmlmb3JtKDAuMSwgMC45KSArIHJuZy51bmlmb3JtKC0wLjYsIDAuNiwgbGVuZ3RoKSAvIDQwLjAKICAgIHYgPSBybmcudW5pZm9ybSgwLjk1LCAxLjA1KSArIHJuZy51bmlmb3JtKC0yLCAyLCBsZW5ndGgpIC8gMjIwLjAKICAgIGxlYWsgPSBybmcudW5pZm9ybSgyLCAyNSkgKyBybmcudW5pZm9ybSgtMiwgMiwgbGVuZ3RoKQogICAgdGVtcCA9IHJuZy51bmlmb3JtKDIwLCA0MCkgKyBybmcudW5pZm9ybSgtMC4zLCAwLjMsIGxlbmd0aCkKICAgIHJldHVybiBbaSwgdiwgbGVhaywgdGVtcF0KCgpkZWYgdWNpX2xpa2Uoc291cmNlLCBybmcsIGxlbmd0aCk6CiAgICBpLCB2ID0gc291cmNlLnNlY29uZHMocm5nLCBsZW5ndGgpCiAgICBpID0gaSAqIHJuZy51bmlmb3JtKDAuNSwgMi4wKQogICAgdiA9IHYgKiBybmcudW5pZm9ybSgwLjk3LCAxLjAzKQogICAgaWYgcm5nLnJhbmRvbSgpIDwgMC4zOgogICAgICAgIGF0ID0gaW50KHJuZy5pbnRlZ2VycygwLCBsZW5ndGgpKQogICAgICAgIGR1ciA9IGludChybmcuaW50ZWdlcnMoMTAsIDIwMCkpCiAgICAgICAgaVthdDphdCArIGR1cl0gKz0gcm5nLnVuaWZvcm0oMC4wNSwgMC40KQogICAgaSA9IG5wLmNsaXAoaSAqICgxICsgcm5nLm5vcm1hbCgwLCAwLjAyLCBsZW5ndGgpKSwgMCwgTm9uZSkKICAgIHYgPSB2ICogKDEgKyBybmcubm9ybWFsKDAsIDAuMDAzLCBsZW5ndGgpKQogICAgbGVhayA9IG5wLmNsaXAocm5nLnVuaWZvcm0oMSwgMjUpICsgMy4wICogaSArIHJuZy51bmlmb3JtKC0yLCAyLCBsZW5ndGgpLCAwLCBOb25lKQogICAgdGVtcCA9IHRoZXJtYWwoaSwgcm5nKSArIHJuZy51bmlmb3JtKC0wLjMsIDAuMywgbGVuZ3RoKQogICAgcmV0dXJuIFtpLCB2LCBsZWFrLCB0ZW1wXQoKCmRlZiBsYWJlbHMoaSwgdiwgbGVhaywgdGVtcCk6CiAgICBjb25kID0gbnAuc3RhY2soW2kgPj0gT1ZFUkxPQURfUFUsICh2IDwgVl9MT1cpIHwgKHYgPiBWX0hJR0gpLCBsZWFrID49IExFQUtfTUEsIHRlbXAgPj0gVEVNUF9DXSwgYXhpcz0xKQogICAgcnVuID0gbnAuemVyb3MoY29uZC5zaGFwZSwgZHR5cGU9aW50KQogICAgZm9yIHQgaW4gcmFuZ2UobGVuKGkpKToKICAgICAgICBydW5bdF0gPSBucC53aGVyZShjb25kW3RdLCAocnVuW3QgLSAxXSBpZiB0IGVsc2UgMCkgKyAxLCAwKQogICAgZmF1bHQgPSBydW4gPj0gUEVSU0lTVAogICAgZmF1bHRbOiwgMF0gfD0gaSA+PSBTSE9SVF9QVQogICAgYW55X2ZhdWx0ID0gZmF1bHQuYW55KGF4aXM9MSkKICAgIHByZSA9IG5wLnplcm9zKGxlbihpKSwgZHR5cGU9Ym9vbCkKICAgIHVwY29taW5nID0gbnAubm9uemVybyhhbnlfZmF1bHQpWzBdCiAgICBmb3IgdCBpbiBucC5ub256ZXJvKH5hbnlfZmF1bHQpWzBdOgogICAgICAgIG54dCA9IHVwY29taW5nW25wLnNlYXJjaHNvcnRlZCh1cGNvbWluZywgdCArIDEpXSBpZiBucC5zZWFyY2hzb3J0ZWQodXBjb21pbmcsIHQgKyAxKSA8IGxlbih1cGNvbWluZykgZWxzZSBOb25lCiAgICAgICAgcHJlW3RdID0gbnh0IGlzIG5vdCBOb25lIGFuZCBueHQgLSB0IDw9IEhPUklaT04KICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShbZmF1bHQsIHByZVs6LCBOb25lXV0sIGF4aXM9MSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIGZlYXR1cmVzKGksIHYsIGxlYWssIHRlbXApOgogICAgaSA9IG5wLmNsaXAobnAubmFuX3RvX251bShpKSwgMCwgMjApCiAgICB2ID0gbnAuY2xpcChucC5uYW5fdG9fbnVtKHYsIG5hbj0xLjApLCAwLCAxLjYpCiAgICBsZWFrID0gbnAuY2xpcChucC5uYW5fdG9fbnVtKGxlYWspLCAwLCAyMDAwKQogICAgdGVtcCA9IG5wLmNsaXAobnAubmFuX3RvX251bSh0ZW1wLCBuYW49MjUuMCksIC0yMCwgMTUwKQogICAgcmV0dXJuIG5wLnN0YWNrKFtucC5sb2cxcChpKSwgdiwgbnAubG9nMXAobGVhaykgLyA3LjAsIHRlbXAgLyAxMDAuMF0sIGF4aXM9LTEpLmFzdHlwZShucC5mbG9hdDMyKQoKCmRlZiBzZXF1ZW5jZShybmcsIHNvdXJjZSwgZmF1bHRfcmF0ZT0wLjU1KToKICAgIGxlbmd0aCA9IFdJTkRPVyArIEhPUklaT04KICAgIGFwcF9zdHlsZSA9IHJuZy5yYW5kb20oKSA8IDAuMzUKICAgIHJhdyA9IGFwcF9saWtlKHJuZywgbGVuZ3RoKSBpZiBhcHBfc3R5bGUgZWxzZSB1Y2lfbGlrZShzb3VyY2UsIHJuZywgbGVuZ3RoKQogICAgaWYgcm5nLnJhbmRvbSgpIDwgMC41OgogICAgICAgIGRpc3RyYWN0KHJhdywgcm5nLCBhcHBfc3R5bGUpCiAgICBraW5kcyA9IFtdCiAgICBpZiBybmcucmFuZG9tKCkgPCBmYXVsdF9yYXRlOgogICAgICAgIGtpbmRzLmFwcGVuZChGQVVMVFNbaW50KHJuZy5pbnRlZ2VycygwLCA0KSldKQogICAgICAgIGlmIHJuZy5yYW5kb20oKSA8IDAuMTU6CiAgICAgICAgICAgIGtpbmRzLmFwcGVuZChGQVVMVFNbaW50KHJuZy5pbnRlZ2VycygwLCA0KSldKQogICAgICAgIHN0YXJ0ID0gaW50KHJuZy5pbnRlZ2Vycyg1LCBXSU5ET1cgKyBIT1JJWk9OIC0gMTApKQogICAgICAgIGZvciBraW5kIGluIGtpbmRzOgogICAgICAgICAgICBpbmplY3Qoa2luZCwgcmF3LCBybmcsIHN0YXJ0LCBhcHBfc3R5bGUgYW5kIGtpbmQgIT0gIm92ZXJoZWF0IikKICAgIGksIHYsIGxlYWssIHRlbXAgPSByYXcKICAgIHkgPSBsYWJlbHMoaSwgdiwgbGVhaywgdGVtcCkKICAgIHJldHVybiBmZWF0dXJlcyhpLCB2LCBsZWFrLCB0ZW1wKVs6V0lORE9XXSwgeVs6V0lORE9XXSwgbnAuc3RhY2socmF3LCBheGlzPTEpWzpXSU5ET1ddLmFzdHlwZShucC5mbG9hdDMyKQoKCmRlZiBiYXRjaChuLCBzZWVkLCBzb3VyY2UsIGZhdWx0X3JhdGU9MC41NSk6CiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICAgIHhzLCB5cywgcmF3cyA9IHppcCgqKHNlcXVlbmNlKHJuZywgc291cmNlLCBmYXVsdF9yYXRlKSBmb3IgXyBpbiByYW5nZShuKSkpCiAgICByZXR1cm4gbnAuc3RhY2soeHMpLCBucC5zdGFjayh5cyksIG5wLnN0YWNrKHJhd3MpCg=="}
for name, raw in BLOB.items():
    (ROOT / name).write_bytes(base64.b64decode(raw))
uci = UCI_DIR / "household_power_consumption.txt"
if not uci.exists():
    zip_path = Path("/tmp/household_power_consumption.zip")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, zip_path.open("wb") as handle:
        handle.write(response.read())
    with zipfile.ZipFile(zip_path) as archive:
        member = next(name for name in archive.namelist() if name.endswith("household_power_consumption.txt"))
        archive.extract(member, UCI_DIR)
        extracted = UCI_DIR / member
        if extracted != uci:
            uci.parent.mkdir(parents=True, exist_ok=True)
            extracted.replace(uci)
os.environ["SARI_UCI_PATH"] = str(uci)
print("code and UCI ready", uci.stat().st_size)


In [ ]:
from pathlib import Path
import os
dest = Path("/content/sari-client-data/train_.csv")
dest.parent.mkdir(parents=True, exist_ok=True)
if not dest.exists():
    from google.colab import files
    print("ارفع train_.csv")
    picked = files.upload()
    dest.write_bytes(next(iter(picked.values())))
os.environ["SARI_CLIENT_TRAIN"] = str(dest)
print(dest, dest.stat().st_size)


In [ ]:
import os
os.environ["SARI_UCI_PATH"] = "/content/sari-client-data/uci/household_power_consumption.txt"
os.environ["SARI_CLIENT_TRAIN"] = "/content/sari-client-data/train_.csv"
%cd /content/live_model
!python -u train.py


In [ ]:
from google.colab import files
from pathlib import Path
for name in ("metrics.json", "sari_config.json", "sari_lstm.pt"):
    path = Path("/content/live_model") / name
    if path.exists():
        files.download(str(path))
